# Raw ntuple production-batch comparison

**Purpose:** demonstrate, directly on the raw checkout ROOT files (no processed dataframes),
that our input ntuples come from (at least) two different production batches whose
reconstruction outputs differ for physically identical events, and catalog which variables differ.

This was discovered via the run-dependent numuCC-rad-corr fractions in
`open_data_distributions.ipynb`: for the *same truth class* (true numuCC 1$\pi^0$ in FV),
`numucc_pi0_overlay` events land in the 1gNp1mu BDT selection at ~15% while `nu_overlay`
events land there at ~1.5%, and run 4a open data shows ~8x the 1gNp1mu rate per POT of
runs 1/3/4b data.

**The two batches, from the file version tags:**

| batch | version tags | files |
|---|---|---|
| "new" | `v10_04_07_20` (surprise / retuple), `v10_04_07_23` (nuwro) | all `nu_overlay`, runs 1–3 beam on/off, run 4b beam on/off, nuwro |
| "old" | `v10_04_07_09`/`13`/`14`/`15`/`16`, untagged | CCpi0/NCpi0/nue/dirt overlays, del1g/iso1g, run 4a beam on/off, runs 4c/4d/5 beam off |

Note: run 3 vs run 4b **data/EXT files agree** (both are new-batch, v10_04_07_20) — the
run-4b-group discrepancy in the plots comes from the *MC* (old-batch CCpi0/NCpi0 samples).
The data-side difference shows up for run 4a (old-batch data) and runs 4c/4d/5 (old-batch EXT).
Both contrasts are shown below.

All reads are capped at the first 30k entries per file for speed.

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt

base = "/nevis/riverside/data/leehagaman/ngem/data_files"
NN = 30_000  # entries per file

# beam-off (EXT) cosmic files: identical physics everywhere -> any difference is processing
ext_files = {
    "run 1  (new, v10_04_07_20)":  ("checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_off_data_surprise_reco2_hist_1.root", "new"),
    "run 3  (new, v10_04_07_20)":  ("checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_off_data_surprise_reco2_hist_3.root", "new"),
    "run 4b (new, v10_04_07_20)":  ("checkout_MCC9.10_Run4b_v10_04_07_20_BNB_beam_off_metapatch_retuple_retuple_hist.root", "new"),
    "run 4a (old, untagged)":      ("checkout_MCC9.10_Run4a_BNB_beam_off_data_surprise_reco2_hist.root", "old"),
    "run 4c (old, v10_04_07_14)":  ("checkout_MCC9.10_Run4acd5_v10_04_07_14_BNB_beam_off_surprise_reco2_hist_4c.root", "old"),
    "run 5  (old, v10_04_07_14)":  ("checkout_MCC9.10_Run4acd5_v10_04_07_14_BNB_beam_off_surprise_reco2_hist_5.root", "old"),
}

# beam-on (open data) files
data_files = {
    "run 1  (new, v10_04_07_20)":  ("checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_on_data_surprise_reco2_hist_1_5e19opendata.root", "new"),
    "run 3  (new, v10_04_07_20)":  ("checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_on_data_surprise_reco2_hist_3_1e19opendata.root", "new"),
    "run 4b (new, v10_04_07_20)":  ("checkout_MCC9.10_Run4b_v10_04_07_20_BNB_beam_on_metapatch_retuple_retuple_hist_opendata_20700.root", "new"),
    "run 4a (old, untagged)":      ("checkout_MCC9.10_Run4a_BNB_beam_on_data_surprise_reco2_hist_opendata_19550.root", "old"),
}

# variables that fingerprint the batch, with (tree, branch, bins, xlabel)
fingerprint_vars = [
    ("singlephotonana/vertex_tree", "sss_num_unassociated_hits",
     np.linspace(0, 600, 61), "gLEE SSS num unassociated hits"),
    ("singlephotonana/vertex_tree", "trackstub_num_unassociated_hits",
     np.linspace(0, 200, 51), "gLEE trackstub num unassociated hits"),
    ("singlephotonana/vertex_tree", "reco_slice_num",
     np.arange(-0.5, 8.5, 1), "gLEE reco_slice_num"),
    ("wcpselection/T_BDTvars", "p_veto_score",
     np.linspace(0, 6, 61), "WC p_veto_score (cosmic veto BDT)"),
]

def load_branch(filename, tree, branch, stop=NN):
    with uproot.open(f"{base}/{filename}") as f:
        a = f[tree].arrays([branch], library="np", entry_stop=stop)[branch].astype(float)
    return a[np.isfinite(a) & (np.abs(a) < 1e30)]

def compare_files(file_dic, title):
    """Overlay normalized histograms of the fingerprint variables for a set of files.
    new-batch files: solid blue/green; old-batch files: dashed red/orange."""
    new_colors = ["tab:blue", "tab:cyan", "tab:green"]
    old_colors = ["tab:red", "tab:orange", "tab:brown"]
    fig, axs = plt.subplots(2, 2, figsize=(13, 9))
    axs = axs.flatten()
    for iv, (tree, branch, bins, xlabel) in enumerate(fingerprint_vars):
        ni, oi = 0, 0
        print(f"--- {branch} ---")
        for label, (fn, batch) in file_dic.items():
            a = load_branch(fn, tree, branch)
            if batch == "new":
                color, ls = new_colors[ni % 3], "-"; ni += 1
            else:
                color, ls = old_colors[oi % 3], "--"; oi += 1
            axs[iv].hist(np.clip(a, bins[0], bins[-1]), bins=bins, density=True,
                         histtype="step", lw=1.8, ls=ls, color=color, label=label)
            print(f"  {label:28s} n={len(a):6d}  median={np.median(a):8.2f}  mean={a.mean():8.2f}")
        axs[iv].set_xlabel(xlabel)
        axs[iv].set_ylabel("density")
        if "hits" in branch:
            axs[iv].set_yscale("log")
        axs[iv].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## 1. Beam-off (EXT) cosmics

Cosmic rays don't know which production batch processed them, so any difference between these
files is a software/processing difference, not physics. Watch how the histograms cluster by
**batch**, not by run period: run 3 and run 4b (new) lie on top of each other, while run 4a,
4c, and 5 (old) form a second cluster with ~2x the unassociated-hit counts, a lower
`reco_slice_num`, and a shifted WC `p_veto_score`.

In [ ]:
compare_files(ext_files, "Beam-off (EXT) cosmics: new-batch (solid) vs old-batch (dashed) productions")

## 2. Beam-on open data

Same comparison for beam-on data. Run 4a is the only old-batch beam-on file, and its gLEE hit
counts and `reco_slice_num` separate from runs 1/3/4b exactly like the old-batch EXT files do.
(Beam physics is identical in all four periods, so again this is processing.) `p_veto_score`
does *not* separate here — in beam-triggered events it is dominated by genuine in-time activity,
so its batch shift is only visible in the cosmics-only EXT comparison above.

This is why run 4a open data lands in the 1gNp1mu BDT selection at ~51 events / 1e19 POT while
runs 1/3/4b come in at 6–8.

In [ ]:
compare_files(data_files, "Beam-on open data: new-batch (solid) vs old-batch (dashed) productions")

## 3. Simulation: identical truth, different reconstruction outputs

The cleanest MC demonstration uses **run 4c**, where we have both a new-batch inclusive
`nu_overlay` file and an old-batch `CCpi0_overlay` file. Selecting the *same truth class*
from both (true numuCC with exactly 1 primary $\pi^0$, vertex in FV) gives two samples of
statistically identical events: the truth distributions match, but the gLEE hit counts and the
WC cosmic-veto score do not.

Truth comes from `wcpselection/T_eval` + `T_PFeval` (entry-aligned with the other trees;
alignment is asserted below via run/subrun/event).

In [ ]:
mc_new_fn = "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4c.root"
mc_old_fn = "checkout_MCC9.10_Run4b4c4d5_v10_04_07_15_BNB_CCpi0_overlay_surprise_reco2_hist_4c.root"

def load_mc_truth_filtered(filename, stop=NN):
    """Return dict of fingerprint vars + truth_nuEnergy for true numuCC 1pi0 inFV events."""
    with uproot.open(f"{base}/{filename}") as f:
        ev = f["wcpselection/T_eval"].arrays(
            ["run", "subrun", "event", "truth_isCC", "truth_nuPdg", "truth_vtxInside", "truth_nuEnergy"],
            library="np", entry_stop=stop)
        pf = f["wcpselection/T_PFeval"].arrays(["truth_NprimPio"], library="np", entry_stop=stop)
        vt = f["singlephotonana/vertex_tree"].arrays(
            ["run_number", "subrun_number", "event_number",
             "sss_num_unassociated_hits", "trackstub_num_unassociated_hits", "reco_slice_num"],
            library="np", entry_stop=stop)
        bdt = f["wcpselection/T_BDTvars"].arrays(["p_veto_score"], library="np", entry_stop=stop)
    # the per-event trees must be entry-aligned; verify with run/subrun/event
    assert (ev["run"] == vt["run_number"]).all()
    assert (ev["subrun"] == vt["subrun_number"]).all()
    assert (ev["event"] == vt["event_number"]).all()
    mask = ((ev["truth_isCC"] == 1) & (ev["truth_nuPdg"] == 14) &
            (ev["truth_vtxInside"] == 1) & (pf["truth_NprimPio"] == 1))
    out = {"truth_nuEnergy": ev["truth_nuEnergy"][mask]}
    for k in ["sss_num_unassociated_hits", "trackstub_num_unassociated_hits", "reco_slice_num"]:
        out[k] = vt[k][mask].astype(float)
    out["p_veto_score"] = bdt["p_veto_score"][mask].astype(float)
    print(f"{filename.split('BNB_')[-1][:40]:42s} true numuCC 1pi0 inFV: {mask.sum()} / {len(mask)} events")
    return out

mc_new = load_mc_truth_filtered(mc_new_fn)
mc_old = load_mc_truth_filtered(mc_old_fn)

plot_spec = [
    ("truth_nuEnergy", np.linspace(0, 4000, 41), "TRUTH: neutrino energy (MeV)", False),
    ("sss_num_unassociated_hits", np.linspace(0, 1500, 51), "RECO: gLEE SSS num unassociated hits", True),
    ("trackstub_num_unassociated_hits", np.linspace(0, 400, 41), "RECO: gLEE trackstub num unassoc hits", True),
    ("p_veto_score", np.linspace(0, 6, 61), "RECO: WC p_veto_score", False),
]
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
axs = axs.flatten()
for i, (var, bins, xlabel, logy) in enumerate(plot_spec):
    for dic, label, color, ls in [(mc_new, "nu_overlay run 4c (new batch)", "tab:blue", "-"),
                                  (mc_old, "CCpi0_overlay run 4c (old batch)", "tab:red", "--")]:
        a = dic[var]
        a = a[np.isfinite(a) & (np.abs(a) < 1e30)]
        axs[i].hist(np.clip(a, bins[0], bins[-1]), bins=bins, density=True,
                    histtype="step", lw=1.8, color=color, ls=ls, label=label)
        print(f"{var:35s} {label:34s} median={np.median(a):8.2f}  mean={a.mean():8.2f}")
    axs[i].set_xlabel(xlabel)
    axs[i].set_ylabel("density")
    if logy:
        axs[i].set_yscale("log")
    axs[i].legend(fontsize=8)
fig.suptitle("Same truth class (true numuCC 1$\\pi^0$ in FV), same run period (4c):\n"
             "truth agrees, reconstruction outputs differ by production batch")
fig.tight_layout()
plt.show()

## 4. What else differs? Broad scan of all scalar branches

Scan every scalar numeric branch of the gLEE `vertex_tree`, the Pandora
`NeutrinoSelectionFilter` tree, and the WC `T_BDTvars` tree, comparing:

- **signal pair**: EXT run 4b (new) vs EXT run 4c (old) — adjacent run periods, pure cosmics,
  so a difference means processing;
- **null pair**: EXT run 3 vs EXT run 4b (both new) — quantifies ordinary period-to-period
  variation under the *same* processing.

A branch is flagged when its new-vs-old shift (max over median / p90 / mean, after clipping
$|x|<10^{30}$ garbage sentinels) is >10%, and more than 3x its null-pair shift. Truth/weight
branches are skipped (uninitialized garbage in beam-off data).

In [ ]:
SKIP_PREFIXES = ("mctruth", "mcflux", "genie", "photonu", "sim_", "geant", "weight",
                 "true", "truth", "m_flash")
SKIP_EXACT = {"run", "subrun", "event", "run_number", "subrun_number", "event_number",
              "pot_per_subrun", "number_of_events_in_subrun", "pot", "event_weight"}

def scalar_branches(tree):
    out = []
    for b in tree.branches:
        if len(b.branches):
            continue
        intp = b.interpretation
        dt = getattr(intp, "numpy_dtype", None)
        if "AsDtype" in repr(type(intp)) and dt is not None and dt.kind in "iuf" and dt.shape == ():
            name = b.name
            if name in SKIP_EXACT or name.lower().startswith(SKIP_PREFIXES):
                continue
            out.append(name)
    return set(out)

def branch_stats(arr):
    a = arr.astype(float)
    a = a[np.isfinite(a) & (np.abs(a) < 1e30)]
    if len(a) < 1000:
        return None
    return np.median(a), np.percentile(a, 90), a.mean()

def rel_shift(sa, sb):
    """Max relative difference over (median, p90, mean)."""
    best = 0.0
    for x, y in zip(sa, sb):
        denom = max(abs(x), abs(y))
        if denom > 1e-12:
            best = max(best, abs(y - x) / denom)
    return best

def scan_tree(tree_path, fn_new, fn_old, fn_null_a, fn_null_b):
    fs = {k: uproot.open(f"{base}/{v}") for k, v in
          [("new", fn_new), ("old", fn_old), ("na", fn_null_a), ("nb", fn_null_b)]}
    common = sorted(set.intersection(*[scalar_branches(f[tree_path]) for f in fs.values()]))
    arrays = {k: f[tree_path].arrays(common, library="np", entry_stop=NN) for k, f in fs.items()}
    for f in fs.values():
        f.close()
    rows = []
    for v in common:
        stats = {k: branch_stats(arrays[k][v]) for k in arrays}
        if any(s is None for s in stats.values()):
            continue
        sig = rel_shift(stats["new"], stats["old"])
        null = rel_shift(stats["na"], stats["nb"])
        if sig > 0.10 and sig > 3 * null:
            rows.append((sig, null, v, stats["new"], stats["old"]))
    rows.sort(reverse=True)
    print(f"\n=== {tree_path}: {len(rows)} flagged of {len(common)} scalar branches ===")
    print(f"{'shift':>6s} {'null':>6s}  {'branch':50s} {'new (med/p90/mean)':>28s}  {'old (med/p90/mean)':>28s}")
    for sig, null, v, sn, so in rows:
        fmt = lambda s: "/".join(f"{x:.3g}" for x in s)
        print(f"{sig:6.2f} {null:6.2f}  {v:50s} {fmt(sn):>28s}  {fmt(so):>28s}")
    return rows

fn_new  = ext_files["run 4b (new, v10_04_07_20)"][0]
fn_old  = ext_files["run 4c (old, v10_04_07_14)"][0]
fn_na   = ext_files["run 3  (new, v10_04_07_20)"][0]
fn_nb   = fn_new

flagged = {}
for tree_path in ["singlephotonana/vertex_tree", "nuselection/NeutrinoSelectionFilter",
                  "wcpselection/T_BDTvars"]:
    flagged[tree_path] = scan_tree(tree_path, fn_new, fn_old, fn_na, fn_nb)

## 5. What changed on the Wire-Cell side specifically

The broad scan above flags WC branches too; this section pins down where in the WC chain the
batches diverge.

**What did *not* change:** the charge-light matching / event-building stage. `T_eval`
quantities (`match_found`, `match_energy`) agree between batches (run 4b new vs run 4a/4c old
EXT are statistically identical), and the flagship `numu_score` / `nue_score` BDTs are stable
for neutrino events.

**What changed** (pattern-recognition stage outputs):

1. the WC **generic-neutrino-selection rate** on EXT cosmics (`kine_reco_Enu > 0`): ~1.65%
   new vs ~1.45% old, and the selected shower-candidate energy spectrum (`mip_energy`) is much
   softer in old files;
2. the **particle-veto BDT score family** (`p_veto_score` seen in section 1, and for neutrino
   events `pi/mu/el/n/all_veto_*` scores), all shifted lower (more cosmic-like) in old files;
3. the **cosmic-tagger (`cosmict_*`) quantities** — filled flags, angles, and dQ/dx front/end
   distributions differ (this is the origin of the huge `wc_cosmict_2_dQ_dx_front/end` means
   seen in the processed dataframes);
4. **sporadic garbage values** in `vis_2_min_medium_dQ_dx` in old files: identical median/p90
   but a tiny fraction of events carry values of order $10^5$–$10^7$ (mean 0.4 new vs ~440
   old) — this was the very first outlier that exposed the batch difference.

Since gLEE and Pandora hit-level quantities change in the same files, it is ambiguous from the
ntuples alone whether the WC shifts come from a WC software change or from an upstream
signal-processing/hit-finding change that all three reconstruction streams inherit.

(The second cell below re-reads many branches from the two MC files and takes a couple of
minutes.)

In [ ]:
# --- 5a. EXT: WC generic selection rate + shower-candidate energy; matching stage as control ---
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
axs = axs.flatten()
new_colors = ["tab:blue", "tab:cyan", "tab:green"]
old_colors = ["tab:red", "tab:orange", "tab:brown"]

sel_fracs, labels, colors = [], [], []
ni, oi = 0, 0
print(f"{'file':30s} {'sel frac':>9s} {'mip_energy mean':>16s} {'p_veto med':>11s} {'match_found':>12s} {'match_energy':>13s}")
for label, (fn, batch) in ext_files.items():
    with uproot.open(f"{base}/{fn}") as f:
        enu = f["wcpselection/T_KINEvars"].arrays(["kine_reco_Enu"], library="np", entry_stop=NN)["kine_reco_Enu"]
        bdt = f["wcpselection/T_BDTvars"].arrays(["mip_energy", "p_veto_score"], library="np", entry_stop=NN)
        ev = f["wcpselection/T_eval"].arrays(["match_found", "match_energy"], library="np", entry_stop=NN)
    if batch == "new":
        color, ls = new_colors[ni % 3], "-"; ni += 1
    else:
        color, ls = old_colors[oi % 3], "--"; oi += 1
    sel = enu > 0
    mip = bdt["mip_energy"][sel]
    pv = bdt["p_veto_score"]; pv = pv[np.abs(pv) < 1e10]
    me = ev["match_energy"].astype(float); me = me[np.isfinite(me) & (np.abs(me) < 1e10)]
    print(f"{label:30s} {sel.mean():9.4f} {mip.mean():16.1f} {np.median(pv):11.3f} "
          f"{ev['match_found'].astype(float).mean():12.3f} {me.mean():13.1f}")
    sel_fracs.append(sel.mean()); labels.append(label); colors.append(color)
    axs[1].hist(np.clip(mip, 0, 800), bins=np.linspace(0, 800, 33), density=True,
                histtype="step", lw=1.8, ls=ls, color=color, label=label)
    axs[2].hist(np.clip(pv, 0, 6), bins=np.linspace(0, 6, 61), density=True,
                histtype="step", lw=1.8, ls=ls, color=color, label=label)
    axs[3].hist(np.clip(me, 0, 2000), bins=np.linspace(0, 2000, 41), density=True,
                histtype="step", lw=1.8, ls=ls, color=color, label=label)

axs[0].bar(range(len(labels)), sel_fracs, color=colors)
axs[0].set_xticks(range(len(labels)))
axs[0].set_xticklabels([l.split("(")[0] for l in labels], rotation=30, ha="right")
axs[0].set_ylabel("frac. of EXT events with kine_reco_Enu > 0")
axs[0].set_title("WC generic selection rate (EXT)")
axs[1].set_xlabel("WC mip_energy of selected EXT events (MeV)"); axs[1].set_ylabel("density"); axs[1].legend(fontsize=7)
axs[2].set_xlabel("WC p_veto_score (all EXT events)"); axs[2].set_ylabel("density"); axs[2].legend(fontsize=7)
axs[3].set_xlabel("WC match_energy (MeV) — matching-stage CONTROL, agrees"); axs[3].set_ylabel("density"); axs[3].legend(fontsize=7)
fig.suptitle("Wire-Cell in EXT cosmics: selection rate, shower energy and veto score differ;\n"
             "the charge-light matching stage does not")
fig.tight_layout()
plt.show()

In [ ]:
# --- 5b. Same-truth MC (run 4c): which WC T_BDTvars branches shift, + veto/cosmict/garbage plots ---
STOP_MC = {"new": 200_000, "old": 40_000}  # more entries for nu_overlay to build up truth-class stats
mc_fns = {"new": mc_new_fn, "old": mc_old_fn}

def wc_truth_class_arrays(which, branches):
    fn, stop = mc_fns[which], STOP_MC[which]
    with uproot.open(f"{base}/{fn}") as f:
        ev = f["wcpselection/T_eval"].arrays(["truth_isCC", "truth_nuPdg", "truth_vtxInside"],
                                             library="np", entry_stop=stop)
        pf = f["wcpselection/T_PFeval"].arrays(["truth_NprimPio"], library="np", entry_stop=stop)
        mask = ((ev["truth_isCC"] == 1) & (ev["truth_nuPdg"] == 14) &
                (ev["truth_vtxInside"] == 1) & (pf["truth_NprimPio"] == 1))
        bdt = f["wcpselection/T_BDTvars"].arrays(branches, library="np", entry_stop=stop)
    return {v: bdt[v][mask] for v in branches}, int(mask.sum())

# scan all common scalar T_BDTvars branches for the truth class
def wc_scalars(fn):
    with uproot.open(f"{base}/{fn}") as f:
        return scalar_branches(f["wcpselection/T_BDTvars"])

common_wc = sorted(wc_scalars(mc_fns["new"]) & wc_scalars(mc_fns["old"]))
wc_new, n_new = wc_truth_class_arrays("new", common_wc)
wc_old, n_old = wc_truth_class_arrays("old", common_wc)
print(f"true numuCC 1pi0 inFV events: new={n_new}, old={n_old}")

wc_rows = []
for v in common_wc:
    stats = []
    ok = True
    for dic in (wc_new, wc_old):
        a = dic[v].astype(float)
        a = a[np.isfinite(a) & (np.abs(a) < 1e30)]
        if len(a) < 500:
            ok = False; break
        stats.append((np.median(a), np.percentile(a, 90), a.mean()))
    if not ok:
        continue
    d = rel_shift(stats[0], stats[1])
    if d > 0.15:
        wc_rows.append((d, v, stats[0], stats[1]))
wc_rows.sort(reverse=True)
print(f"\n{len(wc_rows)} WC branches shifted >15% for the same truth class (med/p90/mean):")
fmt = lambda s: "/".join(f"{x:.3g}" for x in s)
for d, v, sn, so in wc_rows[:30]:
    print(f"  d={d:5.2f}  {v:45s} new {fmt(sn):>26s}   old {fmt(so):>26s}")

# garbage-value rate in vis_2_min_medium_dQ_dx
for lbl, dic in [("new (nu_overlay 4c)", wc_new), ("old (CCpi0 4c)", wc_old)]:
    a = dic["vis_2_min_medium_dQ_dx"].astype(float)
    print(f"vis_2_min_medium_dQ_dx {lbl:22s} mean={a.mean():10.3g}   frac |x|>100: {(np.abs(a) > 100).mean():.5f}")

wc_plot_spec = [
    ("pi_veto_prim_score", np.linspace(-4, 6, 51), "WC pi_veto_prim_score"),
    ("all_veto_score", np.linspace(-4, 8, 61), "WC all_veto_score"),
    ("cosmict_3_angle_beam", np.linspace(0, 30, 61), "WC cosmict_3_angle_beam"),
    ("cosmict_2_dQ_dx_front", np.linspace(0, 5, 51), "WC cosmict_2_dQ_dx_front"),
]
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
axs = axs.flatten()
for i, (var, bins, xlabel) in enumerate(wc_plot_spec):
    for dic, label, color, ls in [(wc_new, "nu_overlay run 4c (new batch)", "tab:blue", "-"),
                                  (wc_old, "CCpi0_overlay run 4c (old batch)", "tab:red", "--")]:
        a = dic[var].astype(float)
        a = a[np.isfinite(a) & (np.abs(a) < 1e10) & (a > -100)]  # drop -999-type sentinels for shape comparison
        axs[i].hist(np.clip(a, bins[0], bins[-1]), bins=bins, density=True,
                    histtype="step", lw=1.8, color=color, ls=ls, label=label)
    axs[i].set_xlabel(xlabel)
    axs[i].set_ylabel("density")
    axs[i].set_yscale("log")
    axs[i].legend(fontsize=8)
fig.suptitle("Wire-Cell veto scores and cosmic-tagger quantities for the SAME truth class\n"
             "(true numuCC 1$\\pi^0$ in FV, run 4c): sentinel values excluded, shapes still differ")
fig.tight_layout()
plt.show()

## Conclusions

**It is not only the one hit count.** The production batches differ in a family of related
quantities (exact lists print above; the headline items, all verified here on raw files):

- **gLEE second-shower-search / trackstub hit counting**: `sss_num_unassociated_hits` (~2x in
  mean, larger in the tail), `trackstub_num_unassociated_hits` (~3x), their `_below_threshold`
  counterparts, and `trackstub_associated_hits` (filled in new-batch files, all-zero in
  old-batch files). The downstream `sss2d_*`/`sss3d_*` ranked quantities shift accordingly.
  In processed dataframes the `glee_dist_ranked_isolation_num_unassoc_hits_win_*` variables
  carry the same fingerprint (4–9 vs 24–50 at preselection level).
- **gLEE slice counting**: `reco_slice_num` drops from median 2 to 1 (and `reco_slice_objects`,
  `reco_asso_tracks` shift with it).
- **Wire-Cell** (section 5): the matching stage (`match_found`, `match_energy`) and the main
  `numu/nue_score` BDTs are batch-independent, but the pattern-recognition outputs are not —
  the generic-selection rate on EXT (~1.65% vs ~1.45%) and its `mip_energy` spectrum, the whole
  particle-veto score family (`p_veto_score` 2.75 vs 2.30 in EXT; `pi/mu/el/n/all_veto_*`
  shifted ~0.3–0.9 lower in old files for same-truth neutrino events), the cosmic-tagger
  `cosmict_*` quantities (the origin of the wild `wc_cosmict_2_dQ_dx_front/end` means in the
  processed dataframes), and rare $10^5$–$10^7$ garbage values in `vis_2_min_medium_dQ_dx` in
  old files.
- **Pandora reconstruction**: real median shifts in the shower Bragg PID scores
  (`shr_bragg_mip`/`mu`/`pion`: ~0.09 new vs ~0.22–0.25 old), fewer `n_pfps` /
  `n_tracks_contained` in old files, shifted `pi0_dedx*_fit_*` and `_elecclusters_*_charge`
  means, plus branch-filling changes: `_closestNuCosmicDist` (real values in old files, 1e9
  sentinel in new) and `secondshower_U_eigenratio` default values.

Because many of these (or variables correlated with them) are BDT training inputs, and because
the 1$\gamma$ signal training samples (del1g / iso1g) are **all old-batch**, the BDT partially
learned the batch fingerprint as a signal signature. Measured consequences (processed-dataframe
level, training `all_vars_r15_2026_08_30`): same-truth numuCC 1$\pi^0$ events leak into 1gNp1mu
at 15.2% (old-batch CCpi0 sample) vs 1.5% (new-batch nu_overlay); EXT leaks at 0.15–0.21% in
old-batch periods vs ~0% in new-batch; run 4a open data shows ~51 events / 1e19 POT in 1gNp1mu
vs 6–8 for runs 1/3/4b.

**Follow-ups to consider:** track down the exact software change between the productions
(likely a gLEE/common-tools version bump around `v10_04_07_20`); either re-tuple the old-batch
files with the new version or retrain the BDT without the batch-sensitive variables; treat run
4a open data and runs 4c/4d/5 EXT with care until then.